In [1]:
import torch
import torch.nn as nn
import pandas as pd
from tqdm import tqdm
from functools import partial
import joblib
import numpy as np
import faiss
import json

from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from src.data.tokenizers.basic_tokenizer import BasicWordTokenizer, BasicCharTokenizer, CustomWordCharTokenizer
from src.data.datastruct import Sample, Batch
from src.data.collate import collate_func, create_batch, create_samples
from src.models.neural_text_classifier import TicketTextClassifierV01
from src.training.train_neural import train_model
from src.evaluation.neural_eval import evaluate, inference_one
from src.evaluation.metrics import evaluate as cm_evaluate, format_cm
from src.models.retrieval_model import retrieval_model_predict
from src.models.confidence_aware_hybrid_model import ConfidenceAwareHybridModel


c:\Work\Project\ticket-nlp-classification\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# setup

device = "cuda" if torch.cuda.is_available() else "cpu"
DATA_PATH = "../data/raw/all_tickets_processed_improved_v3.csv"
df = pd.read_csv(DATA_PATH)
df.head()
EMBEDDING_DIM = 256

X = df["Document"]
y = df["Topic_group"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, shuffle=True, random_state=2)

with open("../artifacts/rac_corpus_similarity-euclidian_index_v01.json", 'r') as f:
    corpus = json.load(f)

index = faiss.read_index("../artifacts/traindata_similarity_index_v01.index")

retrieval_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)

labelencoder: LabelEncoder = joblib.load("../artifacts/labelencoder_neural_v01.pkl")

tokenizer = CustomWordCharTokenizer()
with open("../artifacts/custom_tokenizer_unicar3-5_v01.json", "r") as f:
    tokenizer.load_vocab(json.load(f))

model = TicketTextClassifierV01(
    vocab_size=len(tokenizer),
    embedding_dim=EMBEDDING_DIM,
    pad_id=tokenizer.get_pad_id(),
    n_classes=len(labelencoder.classes_)
).to(device)
model.load_state_dict(torch.load("../artifacts/neural_model_unicar3-5_v01.pt", weights_only=True))


<All keys matched successfully>

In [3]:
hybrid_model = ConfidenceAwareHybridModel(
    model=model,
    tokenizer=tokenizer,
    classes=labelencoder.classes_,
    device=device,
    retrieval_model=retrieval_model,
    index=index,
    corpus=corpus,
    keys=["Document", "Topic_group"],
    y_key="Topic_group",
)

In [6]:
k = 12
thresholds = [0.75, 0.8, 0.85, 0.9]

In [7]:
y_pred_history = {
    0.75: [],
    0.8: [],
    0.85: [],
    0.9: []
}

for _text in tqdm(X_test.to_list()):
    for threshold in thresholds:
        y_pred_history[threshold].append(hybrid_model.predict(_text, threshold=threshold, k = k, retrieval_weight="score"))
    

100%|██████████| 9568/9568 [08:23<00:00, 19.02it/s]


In [11]:
# threshold = 0.75
format_cm(
    cm_evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history[0.75]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.9125    0.9074    0.9099      1425
Administrative rights     0.8645    0.7614    0.8097       352
           HR Support     0.8744    0.8800    0.8772      2183
             Hardware     0.8492    0.8847    0.8666      2724
     Internal Project     0.9058    0.8160    0.8586       424
        Miscellaneous     0.8376    0.8329    0.8352      1412
             Purchase     0.9386    0.8986    0.9181       493
              Storage     0.9215    0.9099    0.9157       555

             accuracy                         0.8740      9568
            macro avg     0.8880    0.8614    0.8739      9568
         weighted avg     0.8745    0.8740    0.8739      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.907368,0.000702,0.032982,0.037193,0.001404,0.016842,0.001404,0.002105
True: Administrative rights,0.011364,0.761364,0.019886,0.170455,0.000000,0.017045,0.017045,0.002841
True: HR Support,0.012368,0.001832,0.879982,0.061383,0.004123,0.034814,0.000458,0.005039
True: Hardware,0.022026,0.009912,0.038546,0.884728,0.002937,0.031204,0.005140,0.005507
True: Internal Project,0.007075,0.007075,0.066038,0.047170,0.816038,0.051887,0.000000,0.004717
True: Miscellaneous,0.016997,0.001416,0.046034,0.081445,0.009915,0.832861,0.004249,0.007082
True: Purchase,0.000000,0.008114,0.020284,0.050710,0.004057,0.016227,0.898580,0.002028
True: Storage,0.010811,0.001802,0.025225,0.037838,0.001802,0.012613,0.000000,0.909910


In [12]:
# threshold = 0.8
format_cm(
    cm_evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history[0.8]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.9083    0.9102    0.9092      1425
Administrative rights     0.8590    0.7614    0.8072       352
           HR Support     0.8768    0.8736    0.8752      2183
             Hardware     0.8487    0.8836    0.8658      2724
     Internal Project     0.9013    0.8184    0.8578       424
        Miscellaneous     0.8346    0.8364    0.8355      1412
             Purchase     0.9404    0.8966    0.9180       493
              Storage     0.9177    0.9045    0.9111       555

             accuracy                         0.8728      9568
            macro avg     0.8859    0.8606    0.8725      9568
         weighted avg     0.8733    0.8728    0.8727      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.910175,0.000702,0.032281,0.035789,0.001404,0.016140,0.001404,0.002105
True: Administrative rights,0.014205,0.761364,0.017045,0.167614,0.000000,0.019886,0.017045,0.002841
True: HR Support,0.013284,0.001832,0.873568,0.063674,0.005039,0.037105,0.000458,0.005039
True: Hardware,0.023128,0.010279,0.037812,0.883627,0.002937,0.031204,0.004772,0.006241
True: Internal Project,0.007075,0.007075,0.061321,0.049528,0.818396,0.051887,0.000000,0.004717
True: Miscellaneous,0.017705,0.001416,0.043909,0.079320,0.009915,0.836402,0.004249,0.007082
True: Purchase,0.000000,0.008114,0.020284,0.050710,0.004057,0.018256,0.896552,0.002028
True: Storage,0.010811,0.003604,0.027027,0.039640,0.001802,0.012613,0.000000,0.904505


In [13]:
# threshold = 0.85
format_cm(
    cm_evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history[0.85]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.8998    0.9074    0.9036      1425
Administrative rights     0.8438    0.7670    0.8036       352
           HR Support     0.8779    0.8694    0.8736      2183
             Hardware     0.8444    0.8807    0.8622      2724
     Internal Project     0.8964    0.8160    0.8543       424
        Miscellaneous     0.8338    0.8314    0.8326      1412
             Purchase     0.9339    0.8884    0.9106       493
              Storage     0.9174    0.9009    0.9091       555

             accuracy                         0.8694      9568
            macro avg     0.8809    0.8577    0.8687      9568
         weighted avg     0.8699    0.8694    0.8693      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.907368,0.002105,0.030877,0.038596,0.002105,0.015439,0.001404,0.002105
True: Administrative rights,0.017045,0.767045,0.017045,0.159091,0.000000,0.019886,0.017045,0.002841
True: HR Support,0.013743,0.001374,0.869446,0.066880,0.005039,0.037563,0.000458,0.005497
True: Hardware,0.024596,0.011747,0.037078,0.880690,0.002937,0.030470,0.006241,0.006241
True: Internal Project,0.009434,0.007075,0.061321,0.049528,0.816038,0.051887,0.000000,0.004717
True: Miscellaneous,0.020538,0.002125,0.043909,0.081445,0.010623,0.831445,0.003541,0.006374
True: Purchase,0.000000,0.008114,0.020284,0.056795,0.004057,0.020284,0.888438,0.002028
True: Storage,0.014414,0.003604,0.027027,0.037838,0.001802,0.014414,0.000000,0.900901


In [14]:
# threshold = 0.9
format_cm(
    cm_evaluate(y_test.to_list(), [_y[0] for _y in y_pred_history[0.9]]), class_names=list(labelencoder.classes_), normalize=True
)

                       precision    recall  f1-score   support

               Access     0.8998    0.9074    0.9036      1425
Administrative rights     0.8438    0.7670    0.8036       352
           HR Support     0.8778    0.8690    0.8734      2183
             Hardware     0.8444    0.8807    0.8622      2724
     Internal Project     0.8964    0.8160    0.8543       424
        Miscellaneous     0.8332    0.8314    0.8323      1412
             Purchase     0.9339    0.8884    0.9106       493
              Storage     0.9174    0.9009    0.9091       555

             accuracy                         0.8693      9568
            macro avg     0.8808    0.8576    0.8686      9568
         weighted avg     0.8698    0.8693    0.8692      9568



,Pred: Access,Pred: Administrative rights,Pred: HR Support,Pred: Hardware,Pred: Internal Project,Pred: Miscellaneous,Pred: Purchase,Pred: Storage
True: Access,0.907368,0.002105,0.030877,0.038596,0.002105,0.015439,0.001404,0.002105
True: Administrative rights,0.017045,0.767045,0.017045,0.159091,0.000000,0.019886,0.017045,0.002841
True: HR Support,0.013743,0.001374,0.868988,0.066880,0.005039,0.038021,0.000458,0.005497
True: Hardware,0.024596,0.011747,0.037078,0.880690,0.002937,0.030470,0.006241,0.006241
True: Internal Project,0.009434,0.007075,0.061321,0.049528,0.816038,0.051887,0.000000,0.004717
True: Miscellaneous,0.020538,0.002125,0.043909,0.081445,0.010623,0.831445,0.003541,0.006374
True: Purchase,0.000000,0.008114,0.020284,0.056795,0.004057,0.020284,0.888438,0.002028
True: Storage,0.014414,0.003604,0.027027,0.037838,0.001802,0.014414,0.000000,0.900901
